In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor, Compose, Normalize
# --- Unsupervised Learning Libraries / مكتبات التعلم غير الخاضع لإشراف ---
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE # t-Distributed Stochastic Neighbor Embedding (Visualization)
from sklearn.cluster import KMeans

import matplotlib.pyplot as plt
import numpy as np
from collections import Counter


from torchvision.datasets import CIFAR10
from torchvision.transforms.functional import to_tensor
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.nn.functional as F
import torch
from torch.optim import AdamW

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)


In [ ]:
# 3. Create DataLoaders
# 1. TRAINING DATALOADER
# This is our delivery system for the model's "study sessions."
train_loader = DataLoader(
    train_dataset,
    batch_size=32,   # 32 images are grouped together. This balances speed and accuracy.
    shuffle=True,    # CRITICAL: We mix the cards every time so the model doesn't
                     # memorize the order of the digits.
    num_workers=2    # Uses 2 separate CPU sub-processes to load data faster,
                     # preventing the GPU from waiting for data.
)

# 2. TEST DATALOADER
# This is the delivery system for the model's "exam."
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,   # No need to shuffle here; we want a consistent way to
                     # measure performance during evaluation.
    num_workers=2
)

# 3. VERIFYING THE BATCH
# iter() makes the loader a stream, and next() grabs the very first package.
X_batch, y_batch = next(iter(train_loader))






In [ ]:
# 4. Print shape of one batch
# Shape explanation:
# X_batch shape: [32, 1, 28, 28] -> [Batch Size, Channels, Height, Width]
# y_batch shape: [32]           -> 32 corresponding labels (0-9)
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")



In [ ]:


# 1. FETCH A BATCH
# 'iter(train_loader)' turns the loader into an iterable stream.
# 'next()' grabs the very first "crate" of data (e.g., 32 or 64 images).
images, labels = next(iter(train_loader))

# 2. SETUP THE GRID
# We create a 2x3 grid to display the first 6 samples from the batch.
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # 3. DIMENSION SWAPPING (The "Permute" Trick)
    # PyTorch stores images as [Channels, Height, Width] (CHW).
    # Matplotlib expects images as [Height, Width, Channels] (HWC).
    # .permute(1, 2, 0) moves the 0th dimension (Channels) to the end.
    img = images[i].permute(1, 2, 0)

    # 4. PLOT & LABEL
    # plt.imshow handles the pixel-to-color mapping.
    plt.imshow(img)

    # .item() converts a 1-element Tensor into a standard Python number.
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off') # Hides the pixel coordinate numbers for a cleaner look

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:

class NN5Layer(nn.Module):

    def __init__(self, input_dim, hidden_dim, output_dim):
        super(NN5Layer, self).__init__()

        # Layer 1: Input Features -> First Hidden Layer
        # input_dim is usually your total pixels (e.g., 784 for MNIST).
        self.layer1 = nn.Linear(input_dim, hidden_dim)

        # Layers 2, 3, and 4: Hidden Layer -> Hidden Layer
        # These keep the dimension consistent so data can flow through the "deep" part.
        self.layer2 = nn.Linear(hidden_dim, hidden_dim)
        self.layer3 = nn.Linear(hidden_dim, hidden_dim)
        self.layer4 = nn.Linear(hidden_dim, hidden_dim)



        # Activation function for non-linearity
        self.relu = nn.ReLU()

    def forward(self, x):
        # The Forward Pass: Data "z" is transformed by weights,
        # then "a" (activation) applies the ReLU non-linearity.

        a1 = self.relu(self.layer1(x))
        a2 = self.relu(self.layer2(a1))
        a3 = self.relu(self.layer3(a2))
        a4 = self.relu(self.layer4(a3))

        # Final Layer: Output raw scores (Logits)
        z4 = self.layer4(a3)

        # ANSWER: Are we missing an activation?
        # No! If using 'nn.CrossEntropyLoss', it applies Softmax internally.
        # Returning raw Logits is more numerically stable for the math.
        return z4

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
    # 1. PREP MODE
    # model.train() tells layers like Dropout or BatchNorm to act in "learning mode."
    model.train()

    running_loss = 0.0

    # 2. THE BATCH LOOP
    # We iterate through the DataLoader, which serves data in small "bites" (batches).
    for X_batch, y_batch in train_loader:

        # Flatten the image: (Batch, 1, 28, 28) -> (Batch, 784) ([32, 3, 36, 36])
        # We must reshape the 2D grid of pixels into a 1D line for Linear layers.
        X_batch = X_batch.view(X_batch.size(0), -1).to(device)
        y_batch = y_batch.to(device)

        # 3. FORWARD PASS
        # The model takes the input and produces 10 raw scores (logits).
        outputs = model(X_batch)

        # 4. COMPUTE ERROR (Loss)
        # We compare the 10 scores to the actual correct digit (y_batch).
        loss = criterion(outputs, y_batch)

        # 5. THE OPTIMIZATION "DANCE"
        optimizer.zero_grad()   # 1. Reset: Wipe the math from the previous batch.
        loss.backward()         # 2. Calculus: Calculate how much each weight contributed to the error.
        optimizer.step()        # 3. Update: Slightly nudge weights in the right direction.

        running_loss += loss.item()

    # 6. PERFORMANCE SUMMARY
    # Return the average "pain level" (loss) the model felt this epoch.
    avg_loss = running_loss / len(train_loader)

    return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
    # 1. EVALUATION MODE
    # Disables layers like Dropout so the model provides consistent predictions.
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    # 2. DISABLE GRADIENTS
    # This saves massive amounts of memory and speed because we aren't
    # calculating "how to improve"—we are just testing.
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            # Flatten and move to GPU/CPU
            X_batch = X_batch.view(X_batch.size(0), -1).to(device)
            y_batch = y_batch.to(device)

            # 3. FORWARD PASS
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            running_loss += loss.item()

            # 4. GET PREDICTIONS
            # Softmax turns raw scores into probabilities that sum to 100%.
            probabilities = F.softmax(outputs, dim=1)

            # Argmax picks the index of the highest probability (the model's guess).
            predicted = torch.argmax(probabilities, dim=1)

            # 5. COUNT SUCCESSES
            # We compare the 'predicted' vector to the 'actual' labels.
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)

    # 6. FINAL METRICS
    avg_loss = running_loss / len(test_loader)

    # Accuracy is simply (Total Correct / Total Attempted)
    accuracy = correct / total

    return avg_loss, accuracy

In [ ]:
# Task 4: Define device, model, loss, optimizer:
# 1. HARDWARE SELECTION
# We check for a GPU (CUDA). If found, the heavy lifting happens on the graphics card,
# which is significantly faster for deep learning than a standard CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. MODEL PARAMETERS (The Blueprint)
# input_dim: CIFAR-10 images are 32x32 pixels with 3 color channels (Red, Green, Blue).
# To feed them into a Linear layer, we must flatten them into one long line: 3 * 32 * 32 = 3072.
input_dim = 3 * 36 * 36

# ([32, 3, 36, 36])
# hidden_dim: This is a "hyperparameter." 64 is a modest starting point.
# More neurons allow for more complex patterns but increase the risk of overfitting.
hidden_dim = 64

# output_dim: Since CIFAR-10 has 10 categories (airplane, bird, car, etc.),
# the model needs 10 output "scorecards."
output_dim = 10

# 3. INSTANTIATION
# This builds the actual "brain" in memory and moves its weights to the GPU if available.
model = NN5Layer(input_dim, hidden_dim, output_dim).to(device)

# 4. INSPECTION
print("Model Architecture:\n")
print(model)

# This counts every single 'synapse' (weight) and 'bias' in your network.
# It tells you how much memory the model will take and how complex it is.
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:
# TODO: choose number of epochs, increase it if you have gpus :)
num_epochs = 20
# TODO: choose a learning rate (try different values and evaluate results)
learning_rate = 0.001

# TODO: Define criterion (loss function) (hint: what loss do we use for multiclass ?)
criterion = nn.CrossEntropyLoss()
# TODO: Define optimizer(what is updated during training?)
optimizer = AdamW(model.parameters(), learning_rate)

In [ ]:
# Run Training
train_losses = []
val_losses = []
val_accuracies = []

print('Starting Training...')
for epoch in range(num_epochs):
    # Train one epoch
    train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

    # Validate
    val_loss, val_accuracy = validate(model, criterion, test_loader, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.4f}')

print('Training Complete!')

In [ ]:
# Task 2 (Bonus): Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.tight_layout()
plt.show()